In [1]:
# import argparse
# parser = argparse.ArgumentParser()

# parser.add_argument("--fold", type=int, required=True)
# args = parser.parse_args()
# fold = args.fold

We will train the scLEMBAS model alongside 4 baselines:

1) random baseline: scLEMBAS trained on permuted features
2) no adversarial baseline: scLEMBAS but global bias does not receive an adversarial penalty for containing categorical / perturbation information
3) linear baseline: emulating the methods [Ahlmann-Eltze et al](https://doi.org/10.1038/s41592-025-02772-6)
- We calculate G as the PCA on the perturbation-psuedo-bulked `Y_train` to 1 component (from 2 perturbations)

The differences include:
- Because the gene embedding `G` is calculated on TF activity, it cannot be directly used for the perturbation embedding `P`. Instead, `P` is retrieved from the GO Embedding as in [Csendes et al](https://doi.org/10.1186/s12864-025-11600-2)

4) non-linear baseline: RF with GO terms, that outperformed foundation models, as described [here](https://doi.org/10.1186/s12864-025-11600-2). The differences include:
- we add a no perturbation control vector of 0s to the binarized GO input prior to PCA, to allow controls in the training/test and make things more directly comparable. 
- we do not psuedobulk by perturbation, as this would decrease the number of samples to very few. Since a perturbation embedding is used, this will still learn the mean of the output features. 

RF Notes:
- Csendes et al do not include a categorical embedding, making this baseline as simplistic as possible. The outputs can be compared against  scLEMBAS predictions psuedobulked by perturbation (direct comparison) and by cell type + perturbation (RF baseline will not change, scLEMBAS will). 
- Csendes et al get the GO PCA embedding on ALL genes. There is an argument to be made to instead subset to the perturbations present in the dataset (more relevant to scLEMBAS training, will capture variability specific to those perturbations) -- bear in mind, the OOD is unseen cell type x perturbation combinations, so all perturbations present in test are also present in train, so this is not an issue. There is also an argument in favor of running on all genes -- capturing a *global* biology prior. We will stick to what the original manuscript did. 

In assessment, we will also use the training mean as a baseline, as described [here](https://doi.org/10.1186/s12864-025-11600-2). The training mean is the psuedo-bulked mean for a given perturbation (independent of other covariates) present in the training data, and the mean across perturbations is taken. 

In [2]:
import os

import torch

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
import scLEMBAS.utilities as utils

sys.path.insert(1, '../.')
from Kang_utils import initialize_mod_and_trainer, all_data 

sys.path.insert(1, '../../.') 
from notebook_utils import RF_baseline, get_split, pert_psuedobulked_mean, linear_baseline

In [3]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'Kang'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [4]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col) = all_data


In [14]:
# for fold in range(5):
#     fn_linear = os.path.join(data_path, 'interim', '{}_fold{}_linearbaseline_prediction.csv'.format(author, fold))
#     y_pred_linear = linear_baseline(fold = fold, adata = tf_adata, pert_col = pert_col, author = author, seed = seed)
#     y_pred_linear.rename(columns = {'perturbation_control': 'CTRL', 'IFNB1': 'STIM'}, inplace = True)
#     y_pred_linear.to_csv(fn_linear)

In [9]:
for fold in range(5):

    # -------------------- SINGLE-CELL MODELS --------------------
    mod_types = {
        'actual': [True, False],
        'random': [True, True],
        'noadv': [False, False]
    }

    fn_base = os.path.join(data_path, 'processed', '{}_fold{}'.format(author, fold))

    def write_model(mod, trainer, mod_type: str):
        io.write_pickled_object(trainer, fn_base + 'trainer_{}.pickle'.format(mod_type))
        torch.save(mod.state_dict(), fn_base + 'model_{}.pt'.format(mod_type))
        del mod, trainer
        utils.clear_memory()

    # Actual Model, Random Baseline, and No Adversarial Removal Baseline
    for mod_type, mod_args in mod_types.items():
        if not os.path.isfile(fn_base + 'trainer_{}.pickle'.format(mod_type)):
            mod, trainer = initialize_mod_and_trainer(
                fold = fold, 
                adversarial_penalty = mod_args[0], 
                randomize = mod_args[1], 
                seed = seed)
            mod = trainer.train_model(verbose = False)
            write_model(mod, trainer, mod_type = mod_type)

    # -------------------- PSUEDO-BULK MODELS --------------------        

    # RF BASELINE    
    fn_rf = os.path.join(data_path, 'interim', '{}_fold{}_RFbaseline_prediction.csv'.format(author, fold))
    if not os.path.isfile(fn_rf):
        y_pred_rf = RF_baseline(fold = fold, adata = tf_adata, author = author, n_cores = n_cores, seed = seed)
        y_pred_rf.to_csv(fn_rf)

    # linear baseline
    fn_linear = os.path.join(data_path, 'interim', '{}_fold{}_linearbaseline_prediction.csv'.format(author, fold))
    if not os.path.isfile(fn_linear):
        y_pred_linear = linear_baseline(fold = fold, adata = tf_adata, pert_col = pert_col, author = author, seed = seed)
        y_pred_linear.rename(columns = {'perturbation_control': 'CTRL', 'IFNB1': 'STIM'}, inplace = True)
        y_pred_linear.to_csv(fn_linear)


    # mean baseline
    fn_mean = os.path.join(data_path, 'interim', '{}_fold{}_meanbaseline_prediction.csv'.format(author, fold))
    if not os.path.isfile(fn_mean):
        split = get_split(fold = fold, author = author)
        _, y_train_mean = pert_psuedobulked_mean(adata = tf_adata, barcodes = split['train_barcodes'], pert_col = pert_col)
        y_train_mean.to_csv(fn_mean)